### Модуль предварительной обработки данных для задачи распознавания степени утомления

In [2]:
import os
import numpy as np
import librosa
import soundfile as sf
import noisereduce as nr  
from pathlib import Path
from tqdm import tqdm

INPUT_PATH = "D:/Аудио выборка/Аудиосегменты/Данные_утомления"  # Путь к исходной папке со всеми файлами
PROCESSED_PATH = "D:/Аудио выборка/Аудиосегменты/Предобработанные_данные"  # Путь для сохранения обработанных данных

SR = 16000  # целевая частота дискретизации
TARGET_DURATION = 1.0  # целевая длительность в секундах
TARGET_LEN = int(SR * TARGET_DURATION)  # 16000 отсчётов
NOISE_REDUCTION_ENABLED = True  # Включаем шумоподавление
NORMALIZATION_TYPE = 'soft'  # 'soft' или 'hard'
NORMALIZATION_RMS_LEVEL = 0.1  # целевой RMS уровень для мягкой нормализации

def reduce_noise_stationary(audio, sr, noise_sample_duration=0.2):
    """
    Шумоподавление стационарного шума с помощью библиотеки noisereduce.

    Аргументы:
        audio (np.ndarray): Массив аудиоданных (float).
        sr (int): Частота дискретизации аудио.
        noise_sample_duration (float): Длительность (сек) сегмента для оценки шума (по умолчанию 0.2).

    Возвращает:
        np.ndarray: Очищенный аудиосигнал размерности, аналогичной входному массиву.
    """
    noise_sample_len = int(noise_sample_duration * sr)
    if len(audio) > noise_sample_len:
        noise_sample = audio[:noise_sample_len]
        audio_clean = nr.reduce_noise(
            y=audio,
            sr=sr,
            y_noise=noise_sample,
            stationary=True,
            prop_decrease=0.8,
            n_std_thresh_stationary=1.5
        )
        return audio_clean
    return audio

def normalize_audio_soft(audio, target_rms=0.1):
    """
    Мягкая нормализация аудио: приведение общего уровня громкости к заданному RMS.

    Аргументы:
        audio (np.ndarray): Массив аудиоданных (float).
        target_rms (float): Целевое значение RMS для нормализации.

    Возвращает:
        np.ndarray: Нормализованный аудиосигнал.
    """
    rms = np.sqrt(np.mean(audio ** 2))
    if rms > 0:
        gain = target_rms / rms
        gain = min(gain, 3.0)
        return audio * gain
    return audio

def adjust_duration(audio, target_len, sr):
    """
    Приведение аудиосигнала к целевой длительности путём обрезки или дополнения нулями.

    Аргументы:
        audio (np.ndarray): Массив аудиосигнала.
        target_len (int): Желаемая длина сигнала в отсчётах (samples).
        sr (int): Частота дискретизации (для совместимости, не используется внутри функции).

    Возвращает:
        np.ndarray: Массив длиной target_len.
    """
    if len(audio) > target_len:
        start = (len(audio) - target_len) // 2
        return audio[start:start + target_len]
    elif len(audio) < target_len:
        return np.pad(audio, (0, target_len - len(audio)), mode='constant')
    return audio

def preprocess_audio(file_path, target_len=TARGET_LEN, sr=SR):
    """
    Полный пайплайн предобработки одного аудиофайла: загрузка, (опционально) шумоподавление, нормализация и усечение/дополнение до заданной длины.

    Аргументы:
        file_path (str): Путь к аудиофайлу.
        target_len (int): Целевая длина сигнала (в отсчётах).
        sr (int): Целевая частота дискретизации.

    Возвращает:
        np.ndarray or None: Обработанный аудиосигнал либо None в случае ошибки или "тихого" файла.
    """
    try:
        audio, orig_sr = librosa.load(file_path, sr=sr)
        
        if np.max(np.abs(audio)) < 0.01:
            return None
        
        if NOISE_REDUCTION_ENABLED:
            audio = reduce_noise_stationary(audio, sr)
        
        if NORMALIZATION_TYPE == 'hard':
            if np.max(np.abs(audio)) > 0:
                audio = audio / np.max(np.abs(audio))
        else:  # soft
            audio = normalize_audio_soft(audio, target_rms=NORMALIZATION_RMS_LEVEL)
        
        audio = adjust_duration(audio, target_len, sr)
        return audio
        
    except Exception as e:
        print(f"Ошибка при обработке {file_path}: {e}")
        return None

def process_folder_single(input_path, output_path, target_len=TARGET_LEN, sr=SR):
    """
    Обрабатывает все аудиофайлы в папке input_path и сохраняет обработанные копии в output_path
    (без деления на классы).

    Аргументы:
        input_path (str): Путь к входной папке с аудиофайлами.
        output_path (str): Путь к папке для сохранения обработанных файлов.
        target_len (int): Желаемая длина обработанного сигнала.
        sr (int): Целевая частота дискретизации.

    Возвращает:
        tuple: (processed_count (int), error_count (int)) — количество успешно и с ошибкой обработанных файлов.
    """
    os.makedirs(output_path, exist_ok=True)
    
    input_files = list(Path(input_path).glob("*.wav"))
    print(f"\nОбработка файлов: найдено {len(input_files)} файлов")
    
    processed_count = 0
    error_count = 0
    
    for file_path in tqdm(input_files, desc="Обработка файлов"):
        audio_processed = preprocess_audio(str(file_path), target_len, sr)
        
        if audio_processed is not None:
            output_file = os.path.join(output_path, file_path.name)
            sf.write(output_file, audio_processed, sr)
            processed_count += 1
        else:
            error_count += 1
    
    print(f"  Успешно обработано: {processed_count}, ошибок: {error_count}")
    return processed_count, error_count

def preprocess_all_data_single(input_path, output_path):
    """
    Запускает обработку всех аудиофайлов в одной папке (без деления на классы), выводит итоговую статистику.

    Аргументы:
        input_path (str): Путь к исходной папке с файлами.
        output_path (str): Путь для сохранения обработанных файлов.

    Возвращает:
        tuple: (processed (int), errors (int)) — количество успешно и с ошибкой обработанных файлов.
    """
    print("="*60)
    print("ПРЕДОБРАБОТКА АУДИОДАННЫХ")
    print("="*60)
    print(f"Целевая частота дискретизации: {SR} Гц")
    print(f"Целевая длительность: {TARGET_DURATION} сек ({TARGET_LEN} отсчётов)")
    print(f"Шумоподавление: {'Включено' if NOISE_REDUCTION_ENABLED else 'Выключено'}")
    print(f"Тип нормализации: {NORMALIZATION_TYPE.upper()}")
    
    if os.path.exists(input_path):
        processed, errors = process_folder_single(input_path, output_path)
    else:
        print(f"\nПапка не найдена - {input_path}")
        processed, errors = 0, 0
    
    print("\n" + "="*60)
    print("ОБРАБОТКА ЗАВЕРШЕНА")
    print("="*60)
    print(f"Всего обработано файлов: {processed}")
    print(f"Ошибок: {errors}")
    print(f"Результаты сохранены в: {output_path}")
    
    return processed, errors

total_processed, total_errors = preprocess_all_data_single(INPUT_PATH, PROCESSED_PATH)

ПРЕДОБРАБОТКА АУДИОДАННЫХ
Целевая частота дискретизации: 16000 Гц
Целевая длительность: 1.0 сек (16000 отсчётов)
Шумоподавление: Включено
Тип нормализации: SOFT

Обработка файлов: найдено 3211 файлов


Обработка файлов: 100%|██████████| 3211/3211 [01:39<00:00, 32.16it/s]

  Успешно обработано: 3211, ошибок: 0

ОБРАБОТКА ЗАВЕРШЕНА
Всего обработано файлов: 3211
Ошибок: 0
Результаты сохранены в: D:/Аудио выборка/Аудиосегменты/Предобработанные_данные
